# Kiểm thử 2 — Robustness
**DS200.F21.CN2 — AI Face Detection**

Đánh giá độ bền của model dưới JPEG compression, resize, và Gaussian blur.

> **Cách chạy:** Bấm **Run All** (▶▶) — không cần thay đổi gì thêm.
> Thời gian ước tính: **10–15 phút** với GPU T4.

In [29]:
import os

root = "/kaggle/input/datasets/"
for dirpath, dirnames, filenames in os.walk(root):
    depth = dirpath.replace(root, "").count(os.sep)
    if depth < 4:
        indent = "  " * depth
        print(f"{indent}{os.path.basename(dirpath)}/")
        if filenames and depth >= 3:
            print(f"{indent}  → ví dụ: {filenames[0]}")

/
ciplab/
xhlulu/
  140k-real-and-fake-faces/
    real_vs_fake/
      real-vs-fake/
hamzaboulahia/
  hardfakevsrealfaces/
    fake/
    real/
manjilkarki/
  deepfake-and-real-images/
    Dataset/
      Validation/
      Test/
      Train/
thanhlongtrn/
  ds200-checkpoint/


In [30]:
# ── Cài thư viện nếu cần ───────────────────────────────
import subprocess
subprocess.run(["pip", "install", "timm", "albumentations", "--upgrade", "-q"],
               capture_output=True)

import os, json, io, pandas as pd, numpy as np
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from PIL import Image, ImageFilter
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, roc_auc_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm

# ── Reproducibility ────────────────────────────────────
import random
random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# ── Config ─────────────────────────────────────────────
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_PATH = "/kaggle/input/datasets/thanhlongtrn/ds200-checkpoint/best_stage3.pth"
TEST_CSV        = "/kaggle/input/datasets/thanhlongtrn/ds200-checkpoint/test.csv"
BATCH_SIZE      = 64
OUTPUT_DIR      = "/kaggle/working/robustness"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Mapping path local → path Kaggle ───────────────────
DATASET_PATH_MAP = {
    "140k-real-and-fake-faces":     "/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/",
    "deepfake-and-real-images":     "/kaggle/input/datasets/manjilkarki/deepfake-and-real-images",
    "hardfakevsrealfaces":          "/kaggle/input/datasets/hamzaboulahia/hardfakevsrealfaces",
    "real-and-fake-face-detection": "/kaggle/input/datasets/ciplab/real-and-fake-face-detection",
}

def remap_path(local_path: str) -> str:
    for key, kaggle_root in DATASET_PATH_MAP.items():
        if key in local_path:
            suffix = local_path[local_path.index(key) + len(key):]
            return kaggle_root + suffix
    return local_path

print(f"Device : {DEVICE}")
print(f"torch  : {torch.__version__}")
print(f"timm   : {timm.__version__}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")


Device : cuda
torch  : 2.10.0+cu128
timm   : 1.0.27
GPU    : Tesla T4


## Định nghĩa Model & Transform

In [31]:
# ── Định nghĩa Model ───────────────────────────────────

class FaceDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            "efficientnet_b0", pretrained=False, num_classes=0
        )
        self.classifier = nn.Sequential(
            nn.Linear(1280, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1),
        )
    def forward(self, x):
        return self.classifier(self.backbone(x))  # ← sửa head → classifier


def load_model(checkpoint_path: str, device) -> FaceDetector:
    model = FaceDetector().to(device)
    ckpt  = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded checkpoint — epoch {ckpt['epoch']}, val_loss {ckpt['val_loss']:.4f}")
    model.eval()
    return model


cfg           = timm.create_model("efficientnet_b0", pretrained=False).default_cfg
MEAN, STD     = list(cfg["mean"]), list(cfg["std"])
val_transform = A.Compose([
    A.LongestMaxSize(max_size=224),
    A.PadIfNeeded(224, 224, border_mode=0),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

print("Model và transform đã sẵn sàng.")


Model và transform đã sẵn sàng.


## Perturbation Functions

| Tên | Mô tả |
|-----|-------|
| `baseline` | Không biến đổi |
| `jpeg_q70/50/30` | Nén JPEG quality 70, 50, 30 |
| `resize_50/25` | Thu nhỏ 50%, 25% rồi phóng lại |
| `blur_s1/s2` | Gaussian blur σ=1.0, σ=2.0 |

In [32]:
# ── Perturbation functions ─────────────────────────────

def apply_jpeg_compression(img_pil: Image.Image, quality: int) -> Image.Image:
    buf = io.BytesIO()
    img_pil.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB")

def apply_resize_downup(img_pil: Image.Image, scale: float) -> Image.Image:
    w, h    = img_pil.size
    small_w = max(1, int(w * scale))
    small_h = max(1, int(h * scale))
    small   = img_pil.resize((small_w, small_h), Image.BILINEAR)
    return small.resize((w, h), Image.BILINEAR)

def apply_gaussian_blur(img_pil: Image.Image, sigma: float) -> Image.Image:
    return img_pil.filter(ImageFilter.GaussianBlur(radius=sigma))

PERTURBATIONS = {
    "baseline":  lambda img: img,
    "jpeg_q70":  lambda img: apply_jpeg_compression(img, 70),
    "jpeg_q50":  lambda img: apply_jpeg_compression(img, 50),
    "jpeg_q30":  lambda img: apply_jpeg_compression(img, 30),
    "resize_50": lambda img: apply_resize_downup(img, 0.50),
    "resize_25": lambda img: apply_resize_downup(img, 0.25),
    "blur_s1":   lambda img: apply_gaussian_blur(img, 1.0),
    "blur_s2":   lambda img: apply_gaussian_blur(img, 2.0),
}

print(f"Đã định nghĩa {len(PERTURBATIONS)} perturbation.")


Đã định nghĩa 8 perturbation.


## Dataset với Perturbation

In [33]:
# ── Dataset với perturbation ───────────────────────────

class RobustnessDataset(Dataset):
    def __init__(self, df: pd.DataFrame, perturb_fn, transform):
        self.df         = df.reset_index(drop=True)
        self.perturb_fn = perturb_fn
        self.transform  = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        img    = Image.open(row["image_path"]).convert("RGB")
        img    = self.perturb_fn(img)
        img_np = np.array(img)
        img_t  = self.transform(image=img_np)["image"]
        label  = torch.tensor(row["label"], dtype=torch.float32)
        return img_t, label


## Load Model & Test Set

In [34]:
model   = load_model(CHECKPOINT_PATH, DEVICE)
test_df = pd.read_csv(TEST_CSV)

# ── Fix 1: chuyển backslash → forward slash ────────────
test_df["image_path"] = test_df["image_path"].str.replace("\\", "/", regex=False)

# ── Fix 2: remap path local → Kaggle ──────────────────
test_df["image_path"] = test_df["image_path"].apply(remap_path)
# Thêm vào cell Load model, sau dòng remap_path
if test_df["label"].dtype == object:
    test_df["label"] = test_df["label"].map({"Real": 1, "Fake": 0})
    print("Đã chuyển label string → int")

print(test_df["label"].value_counts())
# ── Kiểm tra 3 đường dẫn đầu để xác nhận đúng ─────────
print(test_df["image_path"].head(3).to_string())

# ── Kiểm tra file đầu tiên có tồn tại không ───────────
import os
first = test_df["image_path"].iloc[0]
print(f"\nFile tồn tại: {os.path.exists(first)}")

Loaded checkpoint — epoch 30, val_loss 0.0699
Đã chuyển label string → int
label
0    25019
1    22461
Name: count, dtype: int64
0    /kaggle/input/datasets/xhlulu/140k-real-and-fa...
1    /kaggle/input/datasets/xhlulu/140k-real-and-fa...
2    /kaggle/input/datasets/manjilkarki/deepfake-an...

File tồn tại: True


## Chạy Đánh giá

> 8 perturbation × toàn bộ test set — ước tính 10–15 phút trên GPU T4.

In [19]:
print(test_df["label"].dtype)
print(test_df["label"].unique())

object
['Fake' 'Real']


In [35]:
# ── Chạy đánh giá 8 perturbation ──────────────────────
# Mỗi perturbation chạy toàn bộ test set — tổng ~10-15 phút trên GPU T4

robustness_results = {}

for name, perturb_fn in PERTURBATIONS.items():
    print(f"\n[{name}] đang đánh giá...", flush=True)
    dataset = RobustnessDataset(test_df, perturb_fn, val_transform)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE,
                         num_workers=2, pin_memory=True)

    all_preds, all_probs, all_labels = [], [], []
    model.eval()
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc=name, leave=False):
            logits = model(imgs.to(DEVICE)).squeeze(1)
            probs  = torch.sigmoid(logits).cpu().numpy()
            preds  = (probs >= 0.5).astype(int)
            all_probs.extend(probs.tolist())
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.numpy().tolist())

    y, p, prob = np.array(all_labels), np.array(all_preds), np.array(all_probs)
    robustness_results[name] = {
        "accuracy": float(accuracy_score(y, p)),
        "auc":      float(roc_auc_score(y, prob)),
    }
    print(f"  Accuracy={robustness_results[name]['accuracy']:.4f}  "
          f"AUC={robustness_results[name]['auc']:.4f}")

print("\n✓ Hoàn thành đánh giá tất cả perturbation.")



[baseline] đang đánh giá...


  Accuracy=0.0267  AUC=0.0030

[jpeg_q70] đang đánh giá...


  Accuracy=0.1767  AUC=0.0403

[jpeg_q50] đang đánh giá...


  Accuracy=0.2023  AUC=0.0670

[jpeg_q30] đang đánh giá...


  Accuracy=0.2140  AUC=0.0970

[resize_50] đang đánh giá...


  Accuracy=0.1461  AUC=0.0642

[resize_25] đang đánh giá...


  Accuracy=0.2713  AUC=0.1960

[blur_s1] đang đánh giá...


  Accuracy=0.0955  AUC=0.0224

[blur_s2] đang đánh giá...


  Accuracy=0.2635  AUC=0.1789

✓ Hoàn thành đánh giá tất cả perturbation.


## Lưu Kết quả & Biểu đồ

In [36]:
# ── Lưu kết quả & vẽ biểu đồ ──────────────────────────

# 1. Lưu JSON
with open(f"{OUTPUT_DIR}/robustness_metrics.json", "w") as f:
    json.dump(robustness_results, f, indent=2)

# 2. Chuẩn bị dữ liệu
ORDER    = ["baseline","jpeg_q70","jpeg_q50","jpeg_q30",
            "resize_50","resize_25","blur_s1","blur_s2"]
names    = [n for n in ORDER if n in robustness_results]
acc_vals = [robustness_results[n]["accuracy"] for n in names]
auc_vals = [robustness_results[n]["auc"]      for n in names]
base_acc = robustness_results["baseline"]["accuracy"]
base_auc = robustness_results["baseline"]["auc"]

# 3. Line chart Accuracy
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(names, acc_vals, marker="o", linewidth=2, color="#4C9BE8", label="Accuracy")
ax.axhline(base_acc, linestyle="--", color="#999", linewidth=1,
           label=f"Baseline ({base_acc:.3f})")
for n, v in zip(names, acc_vals):
    ax.annotate(f"{v:.3f}", (n, v), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=8)
ax.set_ylim(max(0, min(acc_vals) - 0.05), 1.02)
ax.set_xlabel("Perturbation"); ax.set_ylabel("Accuracy")
ax.set_title("Robustness Test — Accuracy theo Perturbation")
ax.legend(); ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/robustness_accuracy.png", dpi=150, bbox_inches="tight")
plt.close(fig)

# 4. Line chart AUC
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(names, auc_vals, marker="s", linewidth=2, color="#E87C4C", label="AUC-ROC")
ax.axhline(base_auc, linestyle="--", color="#999", linewidth=1,
           label=f"Baseline ({base_auc:.3f})")
for n, v in zip(names, auc_vals):
    ax.annotate(f"{v:.3f}", (n, v), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=8)
ax.set_ylim(max(0, min(auc_vals) - 0.05), 1.02)
ax.set_xlabel("Perturbation"); ax.set_ylabel("AUC-ROC")
ax.set_title("Robustness Test — AUC-ROC theo Perturbation")
ax.legend(); ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/robustness_auc.png", dpi=150, bbox_inches="tight")
plt.close(fig)

# 5. Bar chart accuracy drop
acc_drops = [base_acc - v for v in acc_vals]
colors    = ["#4CE87C" if d <= 0.02 else "#E8C44C" if d <= 0.05 else "#E84C4C"
             for d in acc_drops]
fig, ax   = plt.subplots(figsize=(11, 5))
bars = ax.bar(names, acc_drops, color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Perturbation")
ax.set_ylabel("Accuracy Drop (so với baseline)")
ax.set_title("Robustness Test — Mức sụt giảm Accuracy")
ax.tick_params(axis='x', rotation=20)
for bar, v in zip(bars, acc_drops):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"{v:.3f}", ha="center", va="bottom", fontsize=9)
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/robustness_drop.png", dpi=150, bbox_inches="tight")
plt.close(fig)

print("\n✓ Đã lưu tất cả kết quả vào:", OUTPUT_DIR)
print("\nDanh sách file:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {f}")



✓ Đã lưu tất cả kết quả vào: /kaggle/working/robustness

Danh sách file:
  robustness_accuracy.png
  robustness_auc.png
  robustness_drop.png
  robustness_metrics.json


## Tải kết quả về máy

In [37]:
# ── Nén output để tải về ──────────────────────────────
import shutil
shutil.make_archive("/kaggle/working/robustness_results", "zip",
                    "/kaggle/working/robustness")
print("✓ Đã tạo: /kaggle/working/robustness_results.zip")
print("  → Vào tab Output bên phải → Download file .zip")


✓ Đã tạo: /kaggle/working/robustness_results.zip
  → Vào tab Output bên phải → Download file .zip
